In [ ]:
from pathlib import Path
import pandas as pd
results_path = Path().resolve().parent / "experiments" 
models = ["L1-Qwen3-8B-Max", 
          "Qwen3-8B",
          "L1-Qwen-1.5B-Exact",
          "TokenSkip-Qwen2",
          "QwQ-32B-thinkprune-iter2k",
          "LCR1_7B"
          ]


In [4]:
df = pd.read_parquet(results_path/ model /"evaluation_summary.parquet")
df.head()

,dataset,accuracy,num_correct,num_total,avg_tokens,total_tokens
0,math-500,0.831663,415,499,1137.488978,567607
1,gsm8k,0.801365,1057,1319,374.433662,493878
2,olympiad,0.385757,260,674,2737.470326,1845055
3,amc,0.800000,32,40,1979.725000,79189


In [2]:
df_math = pd.read_parquet(results_path / model / "aime-250_results.parquet")
df_math.head(50)

,unique_id,problem,solution,generated,expected_value,generated_value,token_count,is_correct
0,15206-aime-250,What is the product of the real roots of the e...,20,Let’s think step by step inside and output th...,20,20,2350,True
1,15207-aime-250,Let $a_n=6^{n}+8^{n}$ . Determine the remainde...,35,Let’s think step by step inside and output th...,35,35,853,True
2,15208-aime-250,What is the largest $2$ -digit prime factor of...,61,Let’s think step by step inside and output th...,61,61,1245,True
3,15209-aime-250,The solid shown has a square base of side leng...,288,Let’s think step by step inside and output th...,288,635.65,757,False
4,15210-aime-250,"In tetrahedron $ABCD$ , edge $AB$ has length 3...",20,Let’s think step by step inside and output th...,20,20,308,True
5,15211-aime-250,"A gardener plants three maple trees, four oaks...",106,Let’s think step by step inside and output th...,106,106,654,True
6,15212-aime-250,"Let $x_1=97$ , and for $n>1$ let $x_n=\frac{n}...",384,Let’s think step by step inside and output th...,384,384,721,True
7,15213-aime-250,When a right triangle is rotated about one leg...,26,Let’s think step by step inside and output th...,26,26,831,True
8,15214-aime-250,"Find $c$ if $a$ , $b$ , and $c$ are positive i...",198,Let’s think step by step inside and output th...,198,198,1222,True
9,15215-aime-250,"A sequence of integers $a_1, a_2, a_3, \ldots$...",986,Let’s think step by step inside and output th...,986,986,1487,True


In [3]:

accuracy = df_math["is_correct"].sum() / df_math.shape[0]
accuracy

np.float64(0.424)

In [4]:
avg_token = df_math["token_count"].mean()

avg_token

np.float64(2543.896)

In [ ]:
from pathlib import Path
import pandas as pd

results_path = Path().resolve().parent / "experiments"
models = [
    "L1-Qwen3-8B-Max",
    "Qwen3-8B",
    "L1-Qwen-1.5B-Exact",
    "TokenSkip-Qwen2",
    "QwQ-32B-thinkprune-iter2k",
    "LCR1_7B",
]

datasets = ["math-500", "gsm8k", "olympiad", "amc", "aime-250"]

rows = []
for model in models:
    model_dir = results_path / model
    for dataset in datasets:
        parquet_file = model_dir / f"{dataset}_results.parquet"
        if not parquet_file.exists():
            continue
        df = pd.read_parquet(parquet_file)
        total = len(df)
        correct = df["is_correct"].sum()
        accuracy = correct / total if total > 0 else 0.0
        avg_tokens = df["token_count"].mean()
        rows.append({
            "model": model,
            "dataset": dataset,
            "accuracy": accuracy,
            "num_correct": correct,
            "num_total": total,
            "avg_tokens": avg_tokens,
        })

summary = pd.DataFrame(rows)
summary

,model,dataset,accuracy,num_correct,num_total,avg_tokens
0,L1-Qwen3-8B-Max,math-500,0.731463,365,499,2429.765531
1,L1-Qwen3-8B-Max,gsm8k,0.805914,1063,1319,2137.841547
2,L1-Qwen3-8B-Max,olympiad,0.344214,232,674,3119.321958
3,L1-Qwen3-8B-Max,amc,0.725000,29,40,2749.200000
4,Qwen3-8B,math-500,0.757515,378,499,5895.448898
5,Qwen3-8B,gsm8k,0.900682,1188,1319,5379.167551
6,Qwen3-8B,olympiad,0.292285,197,674,5954.000000
7,Qwen3-8B,amc,0.625000,25,40,5920.675000
8,Qwen3-8B,aime-250,0.228000,57,250,5995.928000
9,TokenSkip-Qwen2,math-500,0.743487,371,499,874.086172


In [4]:
for model in models:
    model_data = summary[summary["model"] == model]
    if model_data.empty:
        print(f"\n{model}: No results found\n")
        continue
    print(f"\n{'='*70}")
    print(f"Model: {model}")
    print(f"{'='*70}")
    print(f"{'Dataset':<15} {'Accuracy':>12} {'Correct':>10} {'Total':>8} {'Avg Tokens':>12}")
    print(f"{'-'*70}")
    for _, row in model_data.iterrows():
        print(f"{row['dataset']:<15} {row['accuracy']*100:>11.2f}% {row['num_correct']:>10} {row['num_total']:>8} {row['avg_tokens']:>12.1f}")
    total_correct = model_data["num_correct"].sum()
    total_problems = model_data["num_total"].sum()
    total_tokens = (model_data["avg_tokens"] * model_data["num_total"]).sum()
    overall_acc = total_correct / total_problems if total_problems > 0 else 0
    overall_avg = total_tokens / total_problems if total_problems > 0 else 0
    print(f"{'-'*70}")
    print(f"{'OVERALL':<15} {overall_acc*100:>11.2f}% {total_correct:>10} {total_problems:>8} {overall_avg:>12.1f}")
    print(f"{'='*70}")


Model: L1-Qwen3-8B-Max
Dataset             Accuracy    Correct    Total   Avg Tokens
----------------------------------------------------------------------
math-500              73.15%        365      499       2429.8
gsm8k                 80.59%       1063     1319       2137.8
olympiad              34.42%        232      674       3119.3
amc                   72.50%         29       40       2749.2
----------------------------------------------------------------------
OVERALL               66.71%       1689     2532       2466.3

Model: Qwen3-8B
Dataset             Accuracy    Correct    Total   Avg Tokens
----------------------------------------------------------------------
math-500              75.75%        378      499       5895.4
gsm8k                 90.07%       1188     1319       5379.2
olympiad              29.23%        197      674       5954.0
amc                   62.50%         25       40       5920.7
aime-250              22.80%         57      250       5995.9
--